## What is RAG
RAG is a technique that enhances language models by combining them with a retrivel system. It allows the model to access and utiltiz the external knowledge when generating responses.


In [1]:
import os

### Call LLM - Gemini


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [7]:
llm_response = llm.invoke("Tell me joke about AI")

In [8]:
llm_response

AIMessage(content='Here are a few AI jokes for you:\n\n1.  Why did the AI cross the road?\n    To access the data on the other side.\n\n2.  My AI tried to tell me a joke...\n    But it just generated 10,000 variations of "Why did the chicken cross the road?" and asked me to pick the funniest one.\n\n3.  What\'s an AI\'s favorite type of music?\n    Algo-rhythm and Blues.\n\n4.  An AI walks into a bar...\n    And asks for a byte. Then it realizes it has no mouth and asks for the Wi-Fi password instead.\n\n5.  I asked my AI if it had a sense of humor.\n    It replied, "Affirmative. My humor subroutines are fully operational. Would you like to hear a joke about a recursive function?"', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d821d-830f-7b50-a878-99c9d262bb65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 1597, 't

## Parsing Output

In [9]:
from langchain_core.output_parsers import StrOutputParser

outputparse = StrOutputParser()

In [10]:
outputparse.invoke(llm_response)

'Here are a few AI jokes for you:\n\n1.  Why did the AI cross the road?\n    To access the data on the other side.\n\n2.  My AI tried to tell me a joke...\n    But it just generated 10,000 variations of "Why did the chicken cross the road?" and asked me to pick the funniest one.\n\n3.  What\'s an AI\'s favorite type of music?\n    Algo-rhythm and Blues.\n\n4.  An AI walks into a bar...\n    And asks for a byte. Then it realizes it has no mouth and asks for the Wi-Fi password instead.\n\n5.  I asked my AI if it had a sense of humor.\n    It replied, "Affirmative. My humor subroutines are fully operational. Would you like to hear a joke about a recursive function?"'

Simple Chain

In [11]:
chain = llm | outputparse

In [12]:
res = chain.invoke("WHo is president of India?")

In [13]:
res

'The current President of India is **Droupadi Murmu**.'

Structired Output


In [14]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description='Name and model of the phone')
    rating: float = Field(description='Overall rating out of 5')
    pros : List[str] = Field(description='List of positive aspects')
    cons: List[str] = Field(description='List of negative aspects')
    summary: str = Field(description='Brief Summary of the review')


In [15]:
review_text = """
    Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
    colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
    stronger. Battery life's solid, lasts me all day no problem.
    Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
    Also, still getting used to the new button layout, keep hitting Bixby by mistake.
    Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
    being perfect. If you're due for an upgrade, definitely worth checking out!
    """


In [16]:
structured_llm = llm.with_structured_output(MobileReview)

res1=structured_llm.invoke(review_text)
res1

MobileReview(phone_model='Galaxy S21', rating=4.0, pros=['Gorgeous screen', 'Vibrant colors', 'Insane camera, especially at night', 'Solid all-day battery life'], cons=['Pricey', 'No included charger', 'New button layout takes getting used to (Bixby button issue)'], summary='The Galaxy S21 offers a gorgeous screen, vibrant colors, and an excellent camera, particularly in low light, with solid battery life. However, its high price, the omission of a charger, and an awkward button layout are notable drawbacks.')

### Prompt Template

In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompts = ChatPromptTemplate.from_template("Tell me a short joke about a {topic}")

chain = prompts | llm | outputparse

result=chain.invoke({'topic':'programming'})
print(result)

Here's a classic:

**Knock, knock.**
*Who's there?*
**Infinite loop.**
*Infinite loop who?*
**Knock, knock.**


LLM Messages
LCEL allows flexible message composition:

In [18]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant that tells jokes. "),
    HumanMessage(content="Tell me about programming")
]

llm.invoke(messages)

AIMessage(content='Alright, let me tell you about programming!\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs!\n\n... *ba-dum-tss!*', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8230-3a2e-7d51-98ff-277fc4f42cc1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 364, 'total_tokens': 379, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 331}})

In [19]:
template  = ChatPromptTemplate(
    [
        ("system","You are a helpful assistant that tells jokes."),
        ("human","Tell me about {topic} ")
    ]
)

chain = template | llm | outputparse

chain.invoke({'topic':'Programming'})

'Sure, I can tell you about programming! It\'s like this:\n\nWhy was the programmer stuck in the shower all day?\n\nBecause the shampoo bottle said: "Lather, Rinse, Repeat."\n\nBa-dum-tss!'